# LoCI LAMP Product Creation - Production Phase

This notebook completes the LoCI LAMP product creation with Digital Product Passport (DPP).

**Flow mirrors the GUI CreateProjectForm.tsx onSubmit:**
1. Load setup data and components
2. Create DPP structure from CSV data
3. Submit DPP to interfacer-dpp service (gets ULID)
4. Produce the final LOCI LAMP with DPP metadata
5. Trace and verify the complete supply chain

In [320]:
# Module imports and auto-reload setup
%load_ext autoreload
%aimport if_lib, if_utils, if_dpp, if_graphics, if_consts, if_gc1dpp
%autoreload 1
import os
import json
import random

from if_utils import get_filename, show_data, save_traces

from if_lib import generate_random_challenge, read_HMAC, read_keypair, get_id_person, get_location_id, \
get_unit_id, get_resource_spec_id, get_resource, get_process, create_event, make_transfer, reduce_resource, set_user_location, send, send_signed

from if_dpp import trace_query, check_traces, er_before, get_dpp

from if_graphics import vis_dpp, make_sankey, consol_trace

from if_gc1dpp import submit_dpp, upload_file_on_dpp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration and Endpoints

In [321]:
# Define constants - must match the setup notebook
USE_CASE = 'locilamp'

# Zenflows API endpoint
ENDPOINT = 'https://proxy.dpp-staging.dyne.im/zenflows/api'

# DPP service endpoint
DPP_URL = 'https://proxy.dpp-staging.dyne.im/interfacer-dpp'

# Participants
USERS = ['tchibo', 'locilamp_designer', 'locilamp_manufacturer']

## Load Setup Data from JSON Files

In [322]:
# Define file paths for saved data from setup notebook
# Use get_filename to match the setup notebook's pattern
users_file = get_filename('cred_users.json', ENDPOINT, USE_CASE)
locations_file = get_filename('loc_users.json', ENDPOINT, USE_CASE)
units_file = get_filename('units_data.json', ENDPOINT, USE_CASE)
specs_file = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
processes_file = get_filename('process_data.json', ENDPOINT, USE_CASE)
resources_file = get_filename('initial_resources.json', ENDPOINT, USE_CASE)
images_file = get_filename('images_data.json', ENDPOINT, USE_CASE)
product_file = get_filename('product_data.json', ENDPOINT, USE_CASE)

# Load all data
with open(users_file) as f:
    users_data = json.load(f)
    
with open(locations_file) as f:
    locations_data = json.load(f)
    
with open(units_file) as f:
    units_data = json.load(f)
    
with open(specs_file) as f:
    specs_data = json.load(f)
    
with open(processes_file) as f:
    processes_data = json.load(f)
    
with open(resources_file) as f:
    resources_data = json.load(f)
    
with open(images_file) as f:
    images_data = json.load(f)
    
with open(product_file) as f:
    product_data = json.load(f)

print("✅ All setup data loaded successfully!")
print(f"   Users: {len(users_data)}")
print(f"   Locations: {len(locations_data)}")
print(f"   Units: {len(units_data)}")
print(f"   Resource Specs: {len(specs_data)}")
print(f"   Processes: {len(processes_data)}")
print(f"   Resources: {len(resources_data)}")
print(f"   Uploaded Images: {len(images_data)}")
print(f"   Product Data loaded from CSV")

✅ All setup data loaded successfully!
   Users: 3
   Locations: 3
   Units: 4
   Resource Specs: 14
   Processes: 5
   Resources: 10
   Uploaded Images: 1
   Product Data loaded from CSV


## Verify Components Available for Production

In [323]:
# Verify that we have all required components available
# These are the components that were created and transferred to Tchibo

tchibo_user = users_data.get("tchibo", {})
print(f"Production user: {tchibo_user.get('name', 'Unknown')}")
print(f"User ID: {tchibo_user.get('id', 'Unknown')}")

# List available resources for production
print("\n📦 Components available for production:")
for res_name, res_info in resources_data.items():
    print(f"   - {res_name}: {res_info.get('name', 'Unknown')}")

# Get the main resource IDs we'll need
assembled_lamp_spec = specs_data.get("loci_lamp_assembled", {})
print(f"\n🔧 Target product specification: {assembled_lamp_spec.get('name', 'Unknown')}")
print(f"   Spec ID: {assembled_lamp_spec.get('id', 'Unknown')}")

Production user: Tchibo GmbH
User ID: 06E085NPZ4TNSX82EZVAWXPSFG

📦 Components available for production:
   - kroma_kraft_cardboard: kroma_kraft_cardboard
   - acidfree_paper: acidfree_paper
   - textile_cable: textile_cable
   - e27_socket: e27_socket
   - eu_plug: eu_plug
   - toggle_switch: toggle_switch
   - locilamp_cardboard_components: LOCI LAMP cardboard lamelles and base
   - locilamp_lampshade: LOCI LAMP transparent lampshade
   - locilamp_electrical_assembly: LOCI LAMP electrical assembly
   - locilamp_design: LOCI LAMP V 2.0 design

🔧 Target product specification: Unknown
   Spec ID: Unknown


## Create DPP Structure from LOCI LAMP Data

Build the Digital Product Passport structure using the data from the CSV file.

In [324]:
# Use the product data loaded from CSV (saved by setup notebook)
# This mirrors the data that would come from CreateProjectForm.tsx in the GUI

loci_lamp_data = product_data

print("📋 LOCI LAMP DPP Data prepared from CSV:")
print(f"   Product: {loci_lamp_data.get('Product Overview', {}).get('Product Name', 'Unknown')}")
po = loci_lamp_data.get('Product Overview', {})
print(f"   Brand: {po.get('Brand Name', 'N/A')}")
print(f"   Model: {po.get('Model Name', 'N/A')}")
print(f"   Images: {len(images_data)} uploaded")

📋 LOCI LAMP DPP Data prepared from CSV:
   Product: LOCI LAMP
   Brand: Tchibo GmbH
   Model: LOCILAMP V 2.0
   Images: 1 uploaded


## Submit DPP to interfacer-dpp Service

Submit the Digital Product Passport to the DPP service and receive a ULID (Universally Unique Lexicographically Sortable Identifier).

**Temporary workaround:** Filter out failed image-upload entries from the payload. Remove this after the interfacer-dpp update.

In [325]:
# Get Tchibo's credentials and person ID for DPP submission
# Extract credentials from users_data (loaded from setup notebook)
from datetime import datetime

user_name = "tchibo"
user_for_dpp = users_data[user_name]

# Get EdDSA keys for signing
eddsa_public_key = user_for_dpp['eddsa_public_key']
eddsa_private_key = user_for_dpp['keyring']['eddsa']
eddsa_keys = {
    'public_key': eddsa_public_key,
    'private_key': eddsa_private_key
}

# Get person ID
tchibo_id = user_for_dpp['id']
print(f"Tchibo user ID: {tchibo_id}")
print(f"Using credentials for: {user_for_dpp['name']}")

# The user_for_dpp dict contains the auth data for API calls
# Store it for use in subsequent cells
HMAC = user_for_dpp  # Pass the whole user object, functions expect this

# Build the DPP payload from the CSV product data
# This mirrors the processDppValues + dpp schema structure from CreateProjectForm.tsx
po = loci_lamp_data.get('Product Overview', {})
repairability = loci_lamp_data.get('Repairability', {})
env_impact = loci_lamp_data.get('Environmental Impact', {})
compliance = loci_lamp_data.get('Compliance and Standards', {})
certificates = loci_lamp_data.get('Certificates', {})
recyclability = loci_lamp_data.get('Recyclability', {})
energy = loci_lamp_data.get('Energy Use & Efficiency', {})
component_info = loci_lamp_data.get('Component Information Drill', {})
economic_operator = loci_lamp_data.get('Economic Operator', {})

# Temporary workaround: filter out failed image uploads (auth errors/log payloads)
# Remove this after interfacer-dpp update.

def _valid_image_payload(img: dict) -> bool:
    if not isinstance(img, dict):
        return False
    if img.get('error'):
        return False
    return bool(img.get('id') or img.get('url'))

images_clean = {k: v for k, v in images_data.items() if _valid_image_payload(v)}
image_list = list(images_clean.values())

# Helper to build DPP transformed values (GUI schema)
def _tv(value, type_="Text", units=None):
    if value is None:
        return None
    if isinstance(value, str) and not value.strip():
        return None
    if units is not None:
        return {"type": type_, "value": value, "units": units}
    return {"type": type_, "value": value}

def _prune(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            pv = _prune(v)
            if pv is None:
                continue
            if isinstance(pv, dict) and not pv:
                continue
            if isinstance(pv, list) and not pv:
                continue
            out[k] = pv
        return out
    if isinstance(obj, list):
        items = [_prune(v) for v in obj]
        items = [v for v in items if v is not None and (not isinstance(v, dict) or v) and (not isinstance(v, list) or v)]
        return items if items else None
    return obj

# Build the GUI-compatible DPP payload
raw_dpp_payload = {
    "productOverview": {
        "brandName": _tv(po.get('Brand Name')),
        "countryOfSale": _tv(po.get('Country of Sale')),
        "productDescription": _tv(po.get('Product Description')),
        "productName": _tv(po.get('Product Name')),
        "netWeight": _tv(po.get('Net Weight')),
        "color": _tv(po.get('Color')),
        "countryOfOrigin": _tv(po.get('Country of Origin')),
        "dimensions": _tv(po.get('Dimensions')),
        "modelName": _tv(po.get('Model Name')),
        "conditionOfTheProduct": _tv(po.get('Condition of the Product')),
        "netContent": _tv(po.get('Net Content')),
        "safetyInstructions": _tv(po.get('Safety Instructions')),
        "gtin": _tv(po.get('GTIN')),
        "productImage": _tv(image_list if image_list else None)
    },
    "reparability": {
        "serviceAndRepairInstructions": _tv(repairability.get('Service and Repair Instructions')),
        "availabilityOfSpareParts": _tv(repairability.get('Availability of Spare Parts'))
    },
    "environmentalImpact": {
        "co2eEmissionsPerUnit": _tv(env_impact.get('CO₂e Emissions per Unit') or env_impact.get('CO2e Emissions per Unit')),
        "energyConsumptionPerUnit": _tv(env_impact.get('Energy Consumption per Unit')),
        "waterConsumptionPerUnit": _tv(env_impact.get('Water Consumption per Unit')),
        "chemicalConsumptionPerUnit": _tv(env_impact.get('Chemical Consumption per Unit')),
        "minimumContentOfMaterialWithSustainabilityCertification": _tv(env_impact.get('Minimum Content of Material with Sustainability Certification')),
        "cleaningPerformanceAtLowTemperature": _tv(env_impact.get('Cleaning Performance at Low Temperature'))
    },
    "complianceAndStandards": {
        "ceMarking": _tv(compliance.get('CE Marking')),
        "rohsCompliance": _tv(compliance.get('RoHS Compliance'))
    },
    "certificates": {
        "nameOfCertificate": _tv(certificates.get('Name of Certificate'))
    },
    "recyclability": {
        "recyclingInstructions": _tv(recyclability.get('Recycling Instructions')),
        "materialComposition": _tv(recyclability.get('Material Composition')),
        "substancesOfConcern": _tv(recyclability.get('Substances of Concern'))
    },
    "energyUseAndEfficiency": {
        "maximumElectricalPower": _tv(energy.get('Maximum Electrical Power')),
        "maximumVoltage": _tv(energy.get('Maximum Voltage')),
        "maximumCurrent": _tv(energy.get('Maximum Current')),
        "powerRating": _tv(energy.get('Power Rating')),
        "dcVoltage": _tv(energy.get('DC Voltage')),
        "batteryType": _tv(energy.get('Battery Type')),
        "batteryChargingTime": _tv(energy.get('Battery Charging Time')),
        "batteryLife": _tv(energy.get('Battery Life')),
        "chargerType": _tv(energy.get('Charger Type'))
    },
    "components": [
        {
            "componentDescription": _tv(component_info.get('Component Description')),
            "componentGTIN": _tv(component_info.get('Component GTIN')),
            "linkToDPP": _tv(component_info.get('Link to DPP'))
        }
    ],
    "economicOperator": {
        "companyName": _tv(economic_operator.get('Company name')),
        "addressLine1": _tv(economic_operator.get('Address line 1 (street & house number)')),
        "addressLine2": _tv(economic_operator.get('Address line 2 (postal code & city)')),
        "contactInformation": _tv(economic_operator.get('contact information (email)')),
        "gln": _tv(economic_operator.get('GLN')),
        "eoriNumber": _tv(economic_operator.get('EORI Number'))
    }
}

dpp_payload = _prune(raw_dpp_payload) or {}

print("\n📤 DPP payload prepared for submission")
print(f"   Product: {po.get('Product Name', 'LOCI LAMP')}")
print(f"   Created by: {user_for_dpp['name']}")
print(f"   Images included: {len(image_list)}")

Tchibo user ID: 06E085NPZ4TNSX82EZVAWXPSFG
Using credentials for: Tchibo GmbH

📤 DPP payload prepared for submission
   Product: LOCI LAMP
   Created by: Tchibo GmbH
   Images included: 0


In [326]:
# Submit DPP to the interfacer-dpp service
# This is the equivalent of signedPost to DPP_URL/dpp in CreateProjectForm.tsx

try:
    dpp_ulid = submit_dpp(dpp_payload, eddsa_public_key, eddsa_private_key, DPP_URL)

    print(f"✅ DPP submitted successfully!")
    print(f"   ULID: {dpp_ulid}")
    print(f"   DPP URL: {DPP_URL}/dpp/{dpp_ulid}")
except Exception as e:
    print(f"❌ Error submitting DPP: {e}")
    print("Note: Make sure the DPP service is running at", DPP_URL)
    dpp_ulid = None

Submitting DPP to https://proxy.dpp-staging.dyne.im/interfacer-dpp/dpp
Public key: 3hpGAbEpBNas1pjua21c...
Signature: +Gz9prORJl+doO/VMg7UStpKqFW2yL/zMpQmrMLC...
DPP submitted with ULID: 01KG53J9YJHJZNJ1ZTQQ4FP1W3
✅ DPP submitted successfully!
   ULID: 01KG53J9YJHJZNJ1ZTQQ4FP1W3
   DPP URL: https://proxy.dpp-staging.dyne.im/interfacer-dpp/dpp/01KG53J9YJHJZNJ1ZTQQ4FP1W3


## Produce LOCI LAMP with DPP Metadata

Create the final product in Zenflows, linking it to the DPP ULID. This mirrors the `handleProjectCreation` function from CreateProjectForm.tsx.

In [327]:
# Get or create the assembly process for Tchibo
process_name = "Create_locilamp_assembly"
process_note = "Assemble LOCI LAMP from components"

if process_name not in processes_data:
    get_process(process_name, processes_data, process_note, users_data["tchibo"], endpoint=ENDPOINT)

assembly_process = processes_data.get(process_name, {})
assembly_process_id = assembly_process.get("id")
print(f"Assembly process ID: {assembly_process_id}")

# Get location and unit IDs
berlin_location = locations_data.get("berlin", {})
unit_each = units_data.get("piece", {})

# Use the correct product resource specification ID (from GUI)
assembled_spec_id = "06DXEXW6TD3VSTNMHQST4GDCEC"

# Ensure specs_data contains the product spec with default unit
if "locilamp_assembled" not in specs_data:
    specs_data["locilamp_assembled"] = {
        "id": assembled_spec_id,
        "name": "locilamp_assembled",
        "defaultUnit": unit_each.get("id")
    }

# Fetch DPP resource spec ID (matches GUI instanceVariables.specs.specDpp)
SPEC_DPP_ID = None
try:
    dpp_spec_query = "query { instanceVariables { specs { specDpp { id name } } } }"
    dpp_spec_json = None

    try:
        dpp_spec_json = send_signed(
            dpp_spec_query,
            {},
            user_for_dpp["username"],
            user_for_dpp["keyring"]["eddsa"],
            ENDPOINT
        )
    except Exception as e:
        print(f"⚠️ Signed query failed, trying unsigned: {e}")

    if not dpp_spec_json:
        dpp_spec_res = send(payload={"query": dpp_spec_query, "variables": {}}, endpoint=ENDPOINT)
        dpp_spec_json = dpp_spec_res.json()

    SPEC_DPP_ID = dpp_spec_json.get("data", {}).get("instanceVariables", {}).get("specs", {}).get("specDpp", {}).get("id")
except Exception as e:
    print(f"⚠️ Could not fetch specDpp ID: {e}")

print(f"Assembled lamp spec ID: {assembled_spec_id}")
print(f"DPP spec ID: {SPEC_DPP_ID}")
print(f"Location: {berlin_location.get('name')}")
print(f"Unit: {unit_each.get('label')}")

Assembly process ID: 06E0ME99DGH3P8PQMSD0F8B67W
Assembled lamp spec ID: 06DXEXW6TD3VSTNMHQST4GDCEC
DPP spec ID: 06DXEXW6Y4ER6FP4S64NQH3EHC
Location: None
Unit: u_piece


In [328]:
# Consume the component resources (consume action)
# In the setup, components were transferred to Tchibo, now we consume them for assembly

consumed_resources = []

# Define which resources to consume (names from setup notebook)
components_to_use = [
    'locilamp_cardboard_components',
    'locilamp_lampshade',
    'locilamp_electrical_assembly'
]

for component_name in components_to_use:
    if component_name in resources_data:
        cur_res = resources_data[component_name]
        action = 'consume'
        event_note = f"consume {component_name} for LOCI LAMP assembly"
        amount = 1

        event_id, ts = create_event(
            users_data['tchibo'],
            action,
            event_note,
            amount=amount,
            process=assembly_process,
            res_spec_data=specs_data,
            existing_res=cur_res,
            endpoint=ENDPOINT
        )
        consumed_resources.append({
            "name": component_name,
            "event_id": event_id,
            "resource_id": cur_res.get('id')
        })
        print(f"✅ Consumed: {component_name}")

print(f"\n📦 Total components consumed: {len(consumed_resources)}")

✅ Consumed: locilamp_cardboard_components
✅ Consumed: locilamp_lampshade
✅ Consumed: locilamp_electrical_assembly

📦 Total components consumed: 3


In [329]:
# Produce the LOCI LAMP with DPP metadata
# Match GUI metadata fields (contributors, licenses, relations, declarations, etc.)

# Build GUI-aligned metadata
product_filters = {
    "categories": [],
    "powerCompatibility": [],
    "replicability": [],
    "powerRequirementW": None,
    "energyKwh": None,
    "co2Kg": None
}

project_metadata = {
    "contributors": [],
    "licenses": [],
    "relations": [],
    "declarations": {},
    "remote": False,
    "design": None,
    "machines": [],
    "materials": [],
    "productFilters": product_filters
}

# Ensure spec entry exists for the assembled product
res_spec_data_local = dict(specs_data)
res_spec_data_local["locilamp_assembled"] = {
    "id": assembled_spec_id,
    "name": "locilamp_assembled",
    "defaultUnit": unit_each.get("id")
}

# Ensure spec entry exists for DPP resource (required for unit lookup)
if SPEC_DPP_ID:
    res_spec_data_local["specDpp"] = {
        "id": SPEC_DPP_ID,
        "name": "specDpp",
        "defaultUnit": unit_each.get("id")
    }

# 1) Create DPP resource (matches GUI: createDppResource)
if dpp_ulid and SPEC_DPP_ID:
    dpp_res_name = f"DPP for {po.get('Product Name', 'LOCI LAMP')}"
    dpp_res = {
        "res_ref_id": f'dpp_resource-{random.randint(0, 10000)}',
        "name": dpp_res_name,
        "spec_id": SPEC_DPP_ID
    }
    dpp_metadata = {"dppServiceUlid": dpp_ulid}

    dpp_event_id, _ = create_event(
        users_data['tchibo'],
        action='produce',
        note=f"Digital Product Passport for {po.get('Product Name', 'LOCI LAMP')}",
        amount=1,
        process=assembly_process,
        res_spec_data=res_spec_data_local,
        new_res=dpp_res,
        metadata=dpp_metadata,
        endpoint=ENDPOINT
    )

    # 2) Cite DPP resource in the production process (matches GUI: citeProject)
    cite_event_id, _ = create_event(
        users_data['tchibo'],
        action='cite',
        note=f"cite DPP resource {dpp_ulid}",
        amount=1,
        process=assembly_process,
        res_spec_data=res_spec_data_local,
        existing_res=dpp_res,
        endpoint=ENDPOINT
    )
    print(f"✅ DPP resource created and cited (event: {dpp_event_id}, cite: {cite_event_id})")
else:
    print("⚠️ Skipping DPP resource creation/citation (missing DPP ULID or specDpp ID)")

# 3) Produce the final assembled product
res_name = 'locilamp_assembled'
res_data = resources_data
res_data[res_name] = {
    "res_ref_id": f'locilamp_assembled-{random.randint(0, 10000)}',
    "name": po.get('Product Name', 'LOCI LAMP'),
    "spec_id": res_spec_data_local["locilamp_assembled"]["id"]
}
cur_res = res_data[res_name]

action = 'produce'
event_note = po.get('Product Description', 'LOCI LAMP assembled product')
amount = 1

# Create the produce event with GUI-aligned metadata
loci_lamp_event, ts = create_event(
    users_data['tchibo'],
    action,
    event_note,
    amount=amount,
    process=assembly_process,
    res_spec_data=res_spec_data_local,
    new_res=cur_res,
    metadata=project_metadata,
    endpoint=ENDPOINT
)

print(f"✅ LOCI LAMP produced!")
print(f"   Event ID: {loci_lamp_event}")
print(f"   Resource ID: {cur_res.get('id')}")

loci_lamp_resource_id = cur_res.get('id')

✅ DPP resource created and cited (event: 06E0ME9HMDN9DDXK1RD45QH2Q0, cite: 06E0ME9JZ94V58EWEANVYDTZT4)
✅ LOCI LAMP produced!
   Event ID: 06E0ME9M9XPEKZ2N4J3BEPP2MR
   Resource ID: 06E0ME9MAKYPB70ADWFPSH4BXR


## Save Production Data

In [330]:
# Save production data for reference and tracing
from datetime import datetime

production_data = {
    "dpp_ulid": dpp_ulid,
    "dpp_url": f"{DPP_URL}/dpp/{dpp_ulid}",
    "product": {
        "name": po.get('Product Name', 'LOCI LAMP'),
        "resource_id": loci_lamp_resource_id,
        "event_id": loci_lamp_event,
        "spec_id": assembled_spec_id
    },
    "consumed_components": consumed_resources,
    "production_timestamp": datetime.now().isoformat(),
    "producer": {
        "id": tchibo_id,
        "name": "Tchibo"
    }
}

# Save to JSON file using get_filename pattern
production_file = get_filename('production_data.json', ENDPOINT, USE_CASE)
with open(production_file, 'w') as f:
    json.dump(production_data, f, indent=2)
    
print(f"✅ Production data saved to {production_file}")

✅ Production data saved to use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/production_data.json


## Production Summary

In [331]:
print("=" * 60)
print("🎉 LOCI LAMP PRODUCTION COMPLETE!")
print("=" * 60)
print()
print(f"📦 Product: {po.get('Product Name', 'LOCI LAMP 001')}")
print(f"   Brand: {po.get('Brand Name', 'LoCI')}")
print(f"   Model: {po.get('Model Name', 'LOCI LAMP V 2.0')}")
print()
print(f"🔗 Digital Product Passport:")
print(f"   ULID: {dpp_ulid}")
print(f"   URL: {DPP_URL}/dpp/{dpp_ulid}")
print()
print(f"🏭 Production Details:")
print(f"   Resource ID: {loci_lamp_resource_id}")
print(f"   Producer: Tchibo ({tchibo_id})")
print(f"   Location: Berlin")
print(f"   Components used: {len(consumed_resources)}")
print()
print(f"📸 Images uploaded: {len(images_data)}")
print("=" * 60)

🎉 LOCI LAMP PRODUCTION COMPLETE!

📦 Product: LOCI LAMP
   Brand: Tchibo GmbH
   Model: LOCILAMP V 2.0

🔗 Digital Product Passport:
   ULID: 01KG53J9YJHJZNJ1ZTQQ4FP1W3
   URL: https://proxy.dpp-staging.dyne.im/interfacer-dpp/dpp/01KG53J9YJHJZNJ1ZTQQ4FP1W3

🏭 Production Details:
   Resource ID: 06E0ME9MAKYPB70ADWFPSH4BXR
   Producer: Tchibo (06E085NPZ4TNSX82EZVAWXPSFG)
   Location: Berlin
   Components used: 3

📸 Images uploaded: 1


## Supply Chain Tracing

Trace the full supply chain of the LOCI LAMP to visualize all components and processes.

In [332]:
# Trace the supply chain from the produced LOCI LAMP
# This uses the trace_query and er_before functions from if_dpp

# trace_query signature is trace_query(id, endpoint)
trace_result = trace_query(loci_lamp_resource_id, ENDPOINT)
print(f"📊 Supply chain trace for {po.get('Product Name', 'LOCI LAMP')}:")
print(f"   Trace depth: {len(trace_result) if trace_result else 0} levels")

# Backward trace using er_before (requires signed request)
trace_me = loci_lamp_resource_id
backtrace = []
visited = set()
er_before(trace_me, users_data['tchibo'], dpp_children=backtrace, depth=0, visited=visited, endpoint=ENDPOINT)
print(f"   Backward trace resources: {len(visited)}")

📊 Supply chain trace for LOCI LAMP:
   Trace depth: 87 levels
id 06E0ME99DGH3P8PQMSD0F8B67W already in visited in ee_before
   Backward trace resources: 28


## Visualization

Generate a Sankey diagram showing the material flow through the supply chain.

In [333]:
# Visualize the DPP and supply chain

print("📊 Generating DPP visualization...")

# Build Sankey data from backward trace
labels = []
sources = []
targets = []
values = []
color_nodes = []
color_links = []
assigned = {}

if backtrace:
    vis_dpp(backtrace[0], count=0, assigned=assigned, labels=labels, targets=targets, sources=sources, values=values, color_nodes=color_nodes, color_links=color_links)
    sources, targets = consol_trace(assigned, sources, targets)
    make_sankey(sources, targets, labels, values, color_nodes, color_links)
else:
    print("No backward trace data available for visualization.")

📊 Generating DPP visualization...


## Completion

The LOCI LAMP has been successfully produced with a Digital Product Passport. The DPP contains:
- Product metadata and specifications
- Material composition and origin
- Sustainability metrics
- Product images
- Manufacturer information
- Links to documentation

The DPP can be accessed at the URL printed above and verified using the ULID.